In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

CSV_PATH = Path("data/cachacaNER.csv")  # ajuste se necessário
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

df = pd.read_csv(CSV_PATH)

In [3]:
for cand in ["sentence_id", "sentence", "sent_id"]:
    if cand in df.columns:
        SENT_COL = cand
        break
else:
    raise KeyError(f"Coluna de sentença não encontrada em {list(df.columns)}")


def sent_to_record(sent_id, g):
    return {
        "sentence_id": int(sent_id),
        "tokens": g["token"].tolist(),
        "ner_tags": g["tag"].tolist(),
    }


In [4]:
records = [sent_to_record(i, g) for i, g in df.groupby(SENT_COL, sort=False)]
cachaca_full = Dataset.from_list(records)

In [5]:
cachaca_full

Dataset({
    features: ['sentence_id', 'tokens', 'ner_tags'],
    num_rows: 13628
})

In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({lab for sent in cachaca_full["ner_tags"] for lab in sent})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
id2label

{0: 'B-CARACTERISTICA_SENSORIAL_AROMA',
 1: 'B-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 2: 'B-CARACTERISTICA_SENSORIAL_COR',
 3: 'B-CARACTERISTICA_SENSORIAL_SABOR',
 4: 'B-CLASSIFICACAO_BEBIDA',
 5: 'B-EQUIPAMENTO_DESTILACAO',
 6: 'B-GRADUACAO_ALCOOLICA',
 7: 'B-NOME_BEBIDA',
 8: 'B-NOME_LOCAL',
 9: 'B-NOME_ORGANIZACAO',
 10: 'B-NOME_PESSOA',
 11: 'B-PRECO',
 12: 'B-RECIPIENTE_ARMAZENAMENTO',
 13: 'B-TEMPO',
 14: 'B-TEMPO_ARMAZENAMENTO',
 15: 'B-TIPO_MADEIRA',
 16: 'B-VOLUME',
 17: 'I-CARACTERISTICA_SENSORIAL_AROMA',
 18: 'I-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 19: 'I-CARACTERISTICA_SENSORIAL_COR',
 20: 'I-CARACTERISTICA_SENSORIAL_SABOR',
 21: 'I-CLASSIFICACAO_BEBIDA',
 22: 'I-EQUIPAMENTO_DESTILACAO',
 23: 'I-GRADUACAO_ALCOOLICA',
 24: 'I-NOME_BEBIDA',
 25: 'I-NOME_LOCAL',
 26: 'I-NOME_ORGANIZACAO',
 27: 'I-NOME_PESSOA',
 28: 'I-PRECO',
 29: 'I-RECIPIENTE_ARMAZENAMENTO',
 30: 'I-TEMPO',
 31: 'I-TEMPO_ARMAZENAMENTO',
 32: 'I-TIPO_MADEIRA',
 33: 'I-VOLUME',
 34: 'O'}

In [8]:
NUM_LABELS

35

# Splits

In [9]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [10]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [11]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [12]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [13]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [14]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [17]:
# standard_split = split_standard(cachaca_full)
# print('std')
# random_splt = random_splits(cachaca_full)
# print('random')
# heur_len = split_heur_length(cachaca_full)
# print("heur_len")
# heur_rare = split_heur_rare(cachaca_full)
# print("heur_rare")
# advers = split_adversarial_fast(cachaca_full)
# print("advs")
print("loc")
loc = loc_split(cachaca_full)
print("semantic")
semantic = semantic_cluster_split(cachaca_full)
print("reverse")
reverse = reverse_curriculum_split(cachaca_full)

loc


/tmp/ipykernel_122855/4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()


semantic


100%|██████████| 13628/13628 [00:00<00:00, 2725017.87it/s]


reverse


# Experimentos

In [18]:
from sklearn.metrics import f1_score as skl_f1

In [19]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "semantic": semantic_cluster_split,
            "reverse": reverse_curriculum_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []   # p/ seqeval
        flat_preds, flat_labels = [], []   # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro    = skl_f1(flat_labels, flat_preds, average="micro",    zero_division=0)
        f1_macro    = skl_f1(flat_labels, flat_preds, average="macro",    zero_division=0)
        f1_weighted = skl_f1(flat_labels, flat_preds, average="weighted", zero_division=0)

        return {
            **seqeval_metrics,            # overall_precision / recall / f1
            "f1_micro":    f1_micro,
            "f1_macro":    f1_macro,
            "f1_weighted": f1_weighted,
        }



    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none"
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [20]:
trainer, loc_metrics = train_ner_with_split(
    cachaca_full, split="reverse"#, split_kwargs={"ngram": 4}
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 12052.19 examples/s]
/tmp/ipykernel_122855/1065706419.py:166: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.175800,0.965246,"{'precision': 0.3333333333333333, 'recall': 0.037037037037037035, 'f1': 0.06666666666666667, 'number': 27}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 13}","{'precision': 0.8636363636363636, 'recall': 0.25, 'f1': 0.3877551020408163, 'number': 76}","{'precision': 0.5081967213114754, 'recall': 0.5040650406504065, 'f1': 0.5061224489795918, 'number': 123}","{'precision': 0.94, 'recall': 0.8623853211009175, 'f1': 0.8995215311004784, 'number': 109}","{'precision': 0.631578947368421, 'recall': 0.8571428571428571, 'f1': 0.7272727272727273, 'number': 14}","{'precision': 0.1476510067114094, 'recall': 0.6666666666666666, 'f1': 0.24175824175824176, 'number': 33}","{'precision': 0.7038461538461539, 'recall': 0.8798076923076923, 'f1': 0.782051282051282, 'number': 208}","{'precision': 0.8588588588588588, 'recall': 0.8719512195121951, 'f1': 0.8653555219364598, 'number': 328}","{'precision': 0.811965811965812, 'recall': 0.6168831168831169, 'f1': 0.7011070110701108, 'number': 154}","{'precision': 0.8142857142857143, 'recall': 0.9344262295081968, 'f1': 0.8702290076335878, 'number': 61}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 559}","{'precision': 0.937888198757764, 'recall': 0.8435754189944135, 'f1': 0.888235294117647, 'number': 179}","{'precision': 0.9606299212598425, 'recall': 0.9457364341085271, 'f1': 0.9531249999999999, 'number': 129}","{'precision': 0.9144736842105263, 'recall': 0.972027972027972, 'f1': 0.9423728813559322, 'number': 143}","{'precision': 0.8773841961852861, 'recall': 0.9096045197740112, 'f1': 0.8932038834951457, 'number': 354}","{'precision': 0.08549874266554904, 'recall': 0.6710526315789473, 'f1': 0.1516728624535316, 'number': 152}",0.521753,0.626221,0.569233,0.842919,0.842919,0.612904,0.825699
2,0.041100,1.101290,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 27}","{'precision': 1.0, 'recall': 0.38461538461538464, 'f1': 0.5555555555555556, 'number': 13}","{'precision': 0.7368421052631579, 'recall': 0.18421052631578946, 'f1': 0.29473684210526313, 'number': 76}","{'precision': 0.6911764705882353, 'recall': 0.3821138211382114, 'f1': 0.49214659685863876, 'number': 123}","{'precision': 0.8241758241758241, 'recall': 0.6880733944954128, 'f1': 0.7500000000000001, 'number': 109}","{'precision': 0.631578947368421, 'recall': 0.8571428571428571, 'f1': 0.7272727272727273, 'number': 14}","{'precision': 0.6285714285714286, 'recall': 0.6666666666666666, 'f1': 0.6470588235294118, 'number': 33}","{'precision': 0.7160493827160493, 'recall': 0.8365384615384616, 'f1': 0.7716186252771619, 'number': 208}","{'precision': 0.8902439024390244, 'recall': 0.8902439024390244, 'f1': 0.8902439024390244, 'number': 328}","{'precision': 0.7972027972027972, 'recall': 0.7402597402597403, 'f1': 0.7676767676767676, 'number': 154}","{'precision': 0.9827586206896551, 'recall': 0.9344262295081968, 'f1': 0.9579831932773109, 'number': 61}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 559}","{'precision': 0.949685534591195, 'recall': 0.8435754189944135, 'f1': 0.8934911242603552, 'number': 179}","{'precision': 0.9461538461538461, 'recall': 0.9534883720930233, 'f1': 0.9498069498069497, 'number': 129}","{'precision': 0.9172413793103448, 'recall': 0.9300699300699301, 'f1': 0.923611111111111, 'number': 143}","{'precision': 0.946927374301676, 'recall': 0.9576271186440678, 'f1': 0.9522471910112359, 'number': 354}","{'precision': 0.08403361344537816, 'recall': 0.6578947368421053, 'f1': 0.14903129657228018, 'number': 152}",0.553774,0.622840,0.586280,0.843584,0.843584

/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [21]:
loc_metrics

{'eval_loss': 0.6027481555938721,
 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.32113821138211385,
  'recall': 0.0933806146572104,
  'f1': 0.1446886446886447,
  'number': 846},
 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.9008264462809917,
  'recall': 0.4448979591836735,
  'f1': 0.5956284153005466,
  'number': 245},
 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.2918149466192171,
  'recall': 0.1902552204176334,
  'f1': 0.2303370786516854,
  'number': 431},
 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.35856573705179284,
  'recall': 0.4462809917355372,
  'f1': 0.3976435935198822,
  'number': 605},
 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.910828025477707,
  'recall': 0.5983263598326359,
  'f1': 0.7222222222222222,
  'number': 478},
 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.8206896551724138,
  'recall': 0.5288888888888889,
  'f1': 0.6432432432432432,
  'number': 225},
 'eval_GRADUACAO_ALCOOLICA': {'precision': 0.5546875,
  'recall': 

In [22]:
loc_metrics.keys()

dict_keys(['eval_loss', 'eval_CARACTERISTICA_SENSORIAL_AROMA', 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA', 'eval_CARACTERISTICA_SENSORIAL_COR', 'eval_CARACTERISTICA_SENSORIAL_SABOR', 'eval_CLASSIFICACAO_BEBIDA', 'eval_EQUIPAMENTO_DESTILACAO', 'eval_GRADUACAO_ALCOOLICA', 'eval_NOME_BEBIDA', 'eval_NOME_LOCAL', 'eval_NOME_ORGANIZACAO', 'eval_NOME_PESSOA', 'eval_PRECO', 'eval_RECIPIENTE_ARMAZENAMENTO', 'eval_TEMPO', 'eval_TEMPO_ARMAZENAMENTO', 'eval_TIPO_MADEIRA', 'eval_VOLUME', 'eval_overall_precision', 'eval_overall_recall', 'eval_overall_f1', 'eval_overall_accuracy', 'eval_f1_micro', 'eval_f1_macro', 'eval_f1_weighted', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'epoch'])

In [23]:
print("LOC test:", loc_metrics["eval_overall_f1"])

LOC test: 0.6677844311377246
